In [1]:
import os
import pandas as pd
import numpy as np
# ── Set your base directory here ──────────────────────────────────────────────
BASE_PATH = '/Users/dhairyabhatt/code_dir/Data analytics projects/world_health/data'   # e.g. r"C:/Users/you/Documents/WHO_Data"
# ──────────────────────────────────────────────────────────────────────────────

def p(folder, filename):
    """Build full file path from folder and filename."""
    return os.path.join(BASE_PATH, folder, filename)

In [5]:
# ── Life Expectancy ───────────────────────────────────────────────────────────
hale_raw        = pd.read_csv(p("LifeExpectencyAndHealthyLifeExpectency", "HALElifeExpectancyAtBirth.csv"))
le_raw          = pd.read_csv(p("LifeExpectencyAndHealthyLifeExpectency", "lifeExpectancyAtBirth.csv"))

# ── Communicable Disease ──────────────────────────────────────────────────────
ntd_raw         = pd.read_csv(p("CommunicableDisease", "interventionAgianstNTDs.csv"))
malaria_raw     = pd.read_csv(p("CommunicableDisease", "incedenceOfMalaria.csv"))
tb_raw          = pd.read_csv(p("CommunicableDisease", "incedenceOfTuberculosis.csv"))

# ── Drinking Water & Sanitation ───────────────────────────────────────────────
sanitation_raw  = pd.read_csv(p("DrinkingWater", "atLeastBasicSanitizationServices.csv"))
safe_san_raw    = pd.read_csv(p("DrinkingWater", "safelySanitization.csv"))
water_raw       = pd.read_csv(p("DrinkingWater", "basicDrinkingWaterServices.csv"))

# ── Healthcare Workforce ──────────────────────────────────────────────────────
doctors_raw     = pd.read_csv(p("HealthWorkForce", "medicalDoctors.csv"))
pharmacists_raw = pd.read_csv(p("HealthWorkForce", "pharmacists.csv"))

# ── UHC / Financial Risk ──────────────────────────────────────────────────────
pop10_raw       = pd.read_csv(p("UHCincludingFinancialRisk", "population10SDG3.8.2.csv"))

print("All files loaded.")

All files loaded.


In [6]:
files = {
    "hale":         hale_raw,
    "le":           le_raw,
    "ntd":          ntd_raw,
    "malaria":      malaria_raw,
    "tb":           tb_raw,
    "sanitation":   sanitation_raw,
    "safe_san":     safe_san_raw,
    "water":        water_raw,
    "doctors":      doctors_raw,
    "pharmacists":  pharmacists_raw,
    "pop10":        pop10_raw,
}

for name, df in files.items():
    null_pct = df.isnull().mean().mul(100).round(1)
    print(f"\n{'─'*55}")
    print(f"  {name}  |  shape: {df.shape}")
    print(f"  columns: {list(df.columns)}")
    print(f"  dtypes:\n{df.dtypes.to_string()}")
    print(f"  null %:\n{null_pct[null_pct > 0].to_string() if null_pct.any() else '  none'}")


───────────────────────────────────────────────────────
  hale  |  shape: (2196, 5)
  columns: ['Location', 'Period', 'Indicator', 'Dim1', 'First Tooltip']
  dtypes:
Location             str
Period             int64
Indicator            str
Dim1                 str
First Tooltip    float64
  null %:
  none

───────────────────────────────────────────────────────
  le  |  shape: (2197, 5)
  columns: ['Location', 'Period', 'Indicator', 'Dim1', 'First Tooltip']
  dtypes:
Location             str
Period             int64
Indicator            str
Dim1                 str
First Tooltip    float64
  null %:
  none

───────────────────────────────────────────────────────
  ntd  |  shape: (1746, 4)
  columns: ['Location', 'Indicator', 'Period', 'First Tooltip']
  dtypes:
Location           str
Indicator          str
Period           int64
First Tooltip    int64
  null %:
  none

───────────────────────────────────────────────────────
  malaria  |  shape: (2033, 4)
  columns: ['Location', 'Indi

In [7]:
for name, df in files.items():
    print(f"{name:15s} → {list(df.columns)}")

hale            → ['Location', 'Period', 'Indicator', 'Dim1', 'First Tooltip']
le              → ['Location', 'Period', 'Indicator', 'Dim1', 'First Tooltip']
ntd             → ['Location', 'Indicator', 'Period', 'First Tooltip']
malaria         → ['Location', 'Indicator', 'Period', 'First Tooltip']
tb              → ['Location', 'Indicator', 'Period', 'First Tooltip']
sanitation      → ['Location', 'Indicator', 'Period', 'Dim1', 'First Tooltip']
safe_san        → ['Location', 'Indicator', 'Period', 'Dim1', 'First Tooltip']
water           → ['Location', 'Period', 'Indicator', 'First Tooltip']
doctors         → ['Location', 'Period', 'Indicator', 'First Tooltip']
pharmacists     → ['Location', 'Period', 'Indicator', 'First Tooltip']
pop10           → ['Location', 'Period', 'Indicator', 'Dim1', 'First Tooltip']


In [8]:
# WHO files consistently use these three column names — confirm from Cell 4 output
# and adjust the rename map below if any file differs.

RENAME_MAP = {
    "Location"      : "country",
    "Period"        : "year",
    "Dim1"          : "dim1",
    "First Tooltip" : "value",
}

def standardise(df):
    return df.rename(columns={k: v for k, v in RENAME_MAP.items() if k in df.columns})

hale_raw        = standardise(hale_raw)
le_raw          = standardise(le_raw)
ntd_raw         = standardise(ntd_raw)
malaria_raw     = standardise(malaria_raw)
tb_raw          = standardise(tb_raw)
sanitation_raw  = standardise(sanitation_raw)
safe_san_raw    = standardise(safe_san_raw)
water_raw       = standardise(water_raw)
doctors_raw     = standardise(doctors_raw)
pharmacists_raw = standardise(pharmacists_raw)
pop10_raw       = standardise(pop10_raw)

print("Column names standardised.")

Column names standardised.


In [9]:
# TB stores values as "183 [150-220]" — extract the central estimate only.

def parse_tb_value(s):
    """Extract leading number from strings like '183 [150-220]' or '12.3'."""
    try:
        return float(str(s).split("[")[0].strip())
    except ValueError:
        return np.nan

tb_raw["value"] = tb_raw["value"].apply(parse_tb_value)

# Verify
print("TB value dtype after parse:", tb_raw["value"].dtype)
print("TB null count after parse: ", tb_raw["value"].isnull().sum())
print(tb_raw["value"].describe())

TB value dtype after parse: float64
TB null count after parse:  0
count    3857.000000
mean      137.027721
std       197.774566
min         0.000000
25%        15.000000
50%        55.000000
75%       188.000000
max      1590.000000
Name: value, dtype: float64


In [10]:
# lifeExpectancyAtBirth.csv goes back to 1920; only 2000-2019 overlaps with HALE.
le_raw = le_raw[le_raw["year"] >= 2000].copy()

print("LE year range after scope:", le_raw["year"].min(), "–", le_raw["year"].max())
print("HALE year range:          ", hale_raw["year"].min(), "–", hale_raw["year"].max())

LE year range after scope: 2000 – 2019
HALE year range:           2000 – 2019


In [11]:
# These files have Urban / Rural / Total rows.
# Mixing all three in an aggregation multiplies counts — always filter explicitly.
# We split into named views here so downstream cells always use the right slice.

# Sanitation
san_total  = sanitation_raw[sanitation_raw["dim1"] == "Total"].copy()
san_urban  = sanitation_raw[sanitation_raw["dim1"] == "Urban"].copy()
san_rural  = sanitation_raw[sanitation_raw["dim1"] == "Rural"].copy()

safe_total = safe_san_raw[safe_san_raw["dim1"] == "Total"].copy()
safe_urban = safe_san_raw[safe_san_raw["dim1"] == "Urban"].copy()
safe_rural = safe_san_raw[safe_san_raw["dim1"] == "Rural"].copy()

# Expenditure (>10% threshold)
pop10_total = pop10_raw[pop10_raw["dim1"] == "Total"].copy()
pop10_urban = pop10_raw[pop10_raw["dim1"] == "Urban"].copy()
pop10_rural = pop10_raw[pop10_raw["dim1"] == "Rural"].copy()

for label, df in [
    ("san_total", san_total), ("san_urban", san_urban), ("san_rural", san_rural),
    ("safe_total", safe_total), ("pop10_total", pop10_total),
]:
    print(f"{label:15s} → {df.shape[0]} rows, {df['country'].nunique()} countries")

san_total       → 3439 rows, 195 countries
san_urban       → 2992 rows, 169 countries
san_rural       → 2937 rows, 166 countries
safe_total      → 1571 rows, 88 countries
pop10_total     → 711 rows, 153 countries


In [12]:
# Workforce data is irregularly reported — not every country has the same latest year.
# For Q25 (pharmacist:doctor ratio) we want each country's most recent observation.

def latest_per_country(df):
    return (
        df.sort_values("year", ascending=False)
          .drop_duplicates(subset="country", keep="first")
          .reset_index(drop=True)
    )

doctors_latest      = latest_per_country(doctors_raw)
pharmacists_latest  = latest_per_country(pharmacists_raw)

print("Doctors   — latest year range:", doctors_latest["year"].min(), "–", doctors_latest["year"].max())
print("Pharmacists — latest year range:", pharmacists_latest["year"].min(), "–", pharmacists_latest["year"].max())
print("Doctors countries:    ", doctors_latest["country"].nunique())
print("Pharmacists countries:", pharmacists_latest["country"].nunique())

Doctors   — latest year range: 2001 – 2018
Pharmacists — latest year range: 1999 – 2018
Doctors countries:     194
Pharmacists countries: 185


In [13]:
# Join on country + year + dim1.
# "Both sexes" row gives the combined figure for global trend analysis.

q2 = pd.merge(
    hale_raw[["country", "year", "dim1", "value"]].rename(columns={"value": "hale"}),
    le_raw[["country", "year", "dim1", "value"]].rename(columns={"value": "le"}),
    on=["country", "year", "dim1"],
    how="inner"
)

q2["hale_yield"] = q2["hale"] / q2["le"]   # Healthy yield ratio (also used in Q23)

print("Q2 shape:", q2.shape)
print("dim1 values:", q2["dim1"].unique())
print("Year range:", q2["year"].min(), "–", q2["year"].max())
print("Countries in join:", q2["country"].nunique())
q2.head(3)

Q2 shape: (2196, 6)
dim1 values: <StringArray>
['Both sexes', 'Male', 'Female']
Length: 3, dtype: str
Year range: 2000 – 2019
Countries in join: 184


,country,year,dim1,hale,le,hale_yield
0,Afghanistan,2019,Both sexes,53.95,63.21,0.853504
1,Afghanistan,2019,Male,54.73,63.29,0.864750
2,Afghanistan,2019,Female,53.15,63.16,0.841514


In [14]:
# Single file, already clean. Isolate 2010 and 2018 endpoints for Q28.

q10 = ntd_raw[["country", "year", "value"]].copy()

q28_endpoints = (
    q10[q10["year"].isin([2010, 2018])]
      .pivot(index="country", columns="year", values="value")
      .rename(columns={2010: "count_2010", 2018: "count_2018"})
      .dropna()
      .reset_index()
)

q28_endpoints["abs_change"]  = q28_endpoints["count_2018"] - q28_endpoints["count_2010"]
q28_endpoints["pct_change"]  = q28_endpoints["abs_change"] / q28_endpoints["count_2010"] * 100
q28_endpoints["burden_q"]    = pd.qcut(q28_endpoints["count_2010"], q=4,
                                        labels=["Q1_lowest", "Q2", "Q3", "Q4_highest"])

print("Q10 shape:", q10.shape)
print("Q28 endpoints shape:", q28_endpoints.shape)
q28_endpoints.sort_values("count_2010", ascending=False).head(5)

Q10 shape: (1746, 3)
Q28 endpoints shape: (193, 6)


year,country,count_2010,count_2018,abs_change,pct_change,burden_q
78,India,837000000.0,697000000.0,-140000000.0,-16.726404,Q4_highest
79,Indonesia,157000000.0,101000000.0,-56000000.0,-35.668790,Q4_highest
125,Nigeria,123000000.0,131000000.0,8000000.0,6.504065,Q4_highest
13,Bangladesh,102000000.0,56339394.0,-45660606.0,-44.765300,Q4_highest
129,Pakistan,69714319.0,31683212.0,-38031107.0,-54.552791,Q4_highest


In [15]:
# Join on country + year + dim1 (Total / Urban / Rural).
# Coverage: ~100 countries (safe sanitation file limit).

q13 = pd.merge(
    san_total[["country", "year", "value"]].rename(columns={"value": "basic_pct"}),
    safe_total[["country", "year", "value"]].rename(columns={"value": "safe_pct"}),
    on=["country", "year"],
    how="inner"
)

q13["quality_gap"] = q13["basic_pct"] - q13["safe_pct"]

# Urban-rural version
q13_ur = pd.merge(
    sanitation_raw[sanitation_raw["dim1"].isin(["Urban", "Rural"])][["country", "year", "dim1", "value"]]
        .rename(columns={"value": "basic_pct"}),
    safe_san_raw[safe_san_raw["dim1"].isin(["Urban", "Rural"])][["country", "year", "dim1", "value"]]
        .rename(columns={"value": "safe_pct"}),
    on=["country", "year", "dim1"],
    how="inner"
)
q13_ur["quality_gap"] = q13_ur["basic_pct"] - q13_ur["safe_pct"]

print("Q13 (Total) shape:", q13.shape, "| countries:", q13["country"].nunique())
print("Q13 (Urban/Rural) shape:", q13_ur.shape)
q13.sort_values("quality_gap", ascending=False).head(5)

Q13 (Total) shape: (1571, 5) | countries: 88
Q13 (Urban/Rural) shape: (2084, 6)


,country,year,basic_pct,safe_pct,quality_gap
53,Andorra,2000,100.00,14.60,85.40
1146,The former Yugoslav Republic of Macedonia,2017,99.12,16.56,82.56
1147,The former Yugoslav Republic of Macedonia,2016,98.70,16.37,82.33
1148,The former Yugoslav Republic of Macedonia,2015,98.14,16.16,81.98
1149,The former Yugoslav Republic of Macedonia,2014,97.57,15.94,81.63


In [16]:
# National trend uses Total only.
# Urban/rural gap uses the split views.

q20_national = pop10_total[["country", "year", "value"]].rename(columns={"value": "catex_pct"}).copy()

q20_ur = pd.concat([
    pop10_urban[["country", "year", "dim1", "value"]],
    pop10_rural[["country", "year", "dim1", "value"]]
]).rename(columns={"value": "catex_pct"})

print("Q20 national shape:", q20_national.shape, "| countries:", q20_national["country"].nunique())
print("Q20 urban/rural shape:", q20_ur.shape)
print("Q20 year range:", q20_national["year"].min(), "–", q20_national["year"].max())

Q20 national shape: (711, 3) | countries: 153
Q20 urban/rural shape: (1182, 4)
Q20 year range: 1985 – 2018


In [17]:
# Step 1: ratio from latest-year workforce snapshots
ratio = pd.merge(
    pharmacists_latest[["country", "value", "year"]].rename(columns={"value": "pharm_per_10k", "year": "pharm_year"}),
    doctors_latest[["country", "value", "year"]].rename(columns={"value": "doc_per_10k", "year": "doc_year"}),
    on="country",
    how="inner"
)

ratio["pharm_doc_ratio"] = ratio["pharm_per_10k"] / ratio["doc_per_10k"].replace(0, np.nan)

# Step 2: join to most recent expenditure figure per country
pop10_latest = latest_per_country(pop10_total.rename(columns={"value": "catex_pct"}))

q25 = pd.merge(
    ratio[["country", "pharm_per_10k", "doc_per_10k", "pharm_doc_ratio"]],
    pop10_latest[["country", "catex_pct"]],
    on="country",
    how="inner"
)

print("Q25 shape:", q25.shape, "| countries:", q25["country"].nunique())
print("Nulls in ratio:", q25["pharm_doc_ratio"].isnull().sum())
q25.describe()

Q25 shape: (149, 5) | countries: 149
Nulls in ratio: 0


,pharm_per_10k,doc_per_10k,pharm_doc_ratio,catex_pct
count,149.000000,149.000000,149.000000,149.000000
mean,3.889591,18.188389,0.241052,8.838993
std,4.161497,16.108717,0.218229,7.939686
min,0.002000,0.140000,0.001105,0.200000
25%,0.380000,3.270000,0.086510,3.130000
50%,2.290000,15.270000,0.183824,6.630000
75%,6.870000,29.920000,0.315619,12.840000
max,19.070000,71.200000,1.044643,54.200000


In [18]:
# Join WASH (water + sanitation Total) to malaria and TB.
# Temporal overlap: 2000–2017 for WASH; malaria up to 2018; TB up to 2019.
# Effective overlap: 2000–2017.

wash = pd.merge(
    water_raw[["country", "year", "value"]].rename(columns={"value": "water_pct"}),
    san_total[["country", "year", "value"]].rename(columns={"value": "sanitation_pct"}),
    on=["country", "year"],
    how="inner"
)

q30_malaria = pd.merge(
    wash,
    malaria_raw[["country", "year", "value"]].rename(columns={"value": "malaria_inc"}),
    on=["country", "year"],
    how="inner"
)

q30_tb = pd.merge(
    wash,
    tb_raw[["country", "year", "value"]].rename(columns={"value": "tb_inc"}),
    on=["country", "year"],
    how="inner"
)

print("Q30 malaria join — shape:", q30_malaria.shape, "| countries:", q30_malaria["country"].nunique())
print("Q30 TB join     — shape:", q30_tb.shape,      "| countries:", q30_tb["country"].nunique())
print("\nNull check — malaria incidence:", q30_malaria["malaria_inc"].isnull().sum())
print("Null check — TB incidence:      ", q30_tb["tb_inc"].isnull().sum())

Q30 malaria join — shape: (1900, 5) | countries: 108
Q30 TB join     — shape: (3426, 5) | countries: 195

Null check — malaria incidence: 0
Null check — TB incidence:       0


In [19]:
# Summary of all analysis-ready dataframes heading into Phase 4.

ready = {
    "Q2  — HALE vs LE":             q2,
    "Q10 — NTD trend":              q10,
    "Q13 — Sanitation quality gap": q13,
    "Q20 — Catastrophic expenditure": q20_national,
    "Q25 — Pharmacist:doctor ratio": q25,
    "Q28 — NTD quartile progress":  q28_endpoints,
    "Q30 — WASH floor (malaria)":   q30_malaria,
    "Q30 — WASH floor (TB)":        q30_tb,
}

print(f"{'Dataset':<35} {'Shape':<15} {'Countries':<12} {'Year Range'}")
print("─" * 75)
for label, df in ready.items():
    countries = df["country"].nunique() if "country" in df.columns else "—"
    yr = f"{int(df['year'].min())}–{int(df['year'].max())}" if "year" in df.columns else "—"
    print(f"{label:<35} {str(df.shape):<15} {str(countries):<12} {yr}")

Dataset                             Shape           Countries    Year Range
───────────────────────────────────────────────────────────────────────────
Q2  — HALE vs LE                    (2196, 6)       184          2000–2019
Q10 — NTD trend                     (1746, 3)       195          2010–2018
Q13 — Sanitation quality gap        (1571, 5)       88           2000–2017
Q20 — Catastrophic expenditure      (711, 3)        153          1985–2018
Q25 — Pharmacist:doctor ratio       (149, 5)        149          —
Q28 — NTD quartile progress         (193, 6)        193          —
Q30 — WASH floor (malaria)          (1900, 5)       108          2000–2017
Q30 — WASH floor (TB)               (3426, 5)       195          2000–2017


In [20]:
# Cell 17 — Export Analysis-Ready Dataframes to CSV

OUTPUT_PATH = os.path.join(BASE_PATH, "phase3_outputs")
os.makedirs(OUTPUT_PATH, exist_ok=True)

exports = {
    "q2_hale_vs_le":          q2,
    "q10_ntd_trend":          q10,
    "q13_sanitation_gap":     q13,
    "q13_sanitation_gap_ur":  q13_ur,
    "q20_catastrophic_exp":   q20_national,
    "q20_catastrophic_exp_ur": q20_ur,
    "q25_pharma_doctor_ratio": q25,
    "q28_ntd_quartile":       q28_endpoints,
    "q30_wash_malaria":       q30_malaria,
    "q30_wash_tb":            q30_tb,
}

for filename, df in exports.items():
    out = os.path.join(OUTPUT_PATH, f"{filename}.csv")
    df.to_csv(out, index=False)
    print(f"Saved: {filename}.csv  |  {df.shape[0]} rows, {df.shape[1]} cols")

Saved: q2_hale_vs_le.csv  |  2196 rows, 6 cols
Saved: q10_ntd_trend.csv  |  1746 rows, 3 cols
Saved: q13_sanitation_gap.csv  |  1571 rows, 5 cols
Saved: q13_sanitation_gap_ur.csv  |  2084 rows, 6 cols
Saved: q20_catastrophic_exp.csv  |  711 rows, 3 cols
Saved: q20_catastrophic_exp_ur.csv  |  1182 rows, 4 cols
Saved: q25_pharma_doctor_ratio.csv  |  149 rows, 5 cols
Saved: q28_ntd_quartile.csv  |  193 rows, 6 cols
Saved: q30_wash_malaria.csv  |  1900 rows, 5 cols
Saved: q30_wash_tb.csv  |  3426 rows, 5 cols
